In [ ]:
%cd /kaggle/working
!git clone https://github.com/yyouretoast/deepfake-detection.git || (cd deepfake-detection && git pull)
%cd /kaggle/working/deepfake-detection
!pip install -q -r requirements.txt

In [ ]:
!accelerate launch --multi_gpu --mixed_precision fp16 --num_processes 2 \
    scripts/train_dual_stream_ddp.py \
    --epochs 5 \
    --batch_size 16 \
    --frequency_backbone resse \
    --hardened \
    --save_path /kaggle/working/dual_stream_best.pth

In [ ]:
!python scripts/evaluate_test_set.py \
    --weights_path /kaggle/working/dual_stream_best.pth \
    --save_calibrated /kaggle/working/dual_stream_calibrated.pth

In [ ]:
!python scripts/train_temporal_head.py \
    --backbone_weights /kaggle/working/dual_stream_best.pth \
    --save_path /kaggle/working/temporal_head_best.pth \
    --epochs 5 \
    --batch_size 8 \
    --seq_len 8

In [ ]:
!python scripts/export_test_predictions.py \
    --checkpoint /kaggle/working/dual_stream_calibrated.pth \
    --output_json /kaggle/working/test_predictions.json

In [ ]:
!python scripts/evaluate_subdomain_breakdown.py \
    --weights_path /kaggle/working/dual_stream_calibrated.pth

In [ ]:
!python scripts/evaluate_robustness.py \
    --checkpoint /kaggle/working/dual_stream_calibrated.pth \
    --output_json /kaggle/working/robustness_results.json

In [ ]:
%%bash
rm -f /kaggle/working/loto_results.json
for fold in deepfakes face2face faceswap neuraltextures celeb; do
    accelerate launch --multi_gpu --mixed_precision fp16 --num_processes 2 \
        scripts/train_loto_experiment.py --holdout $fold --epochs 3 --batch_size 16 --frequency_backbone resse --hardened
done

In [ ]:
!python scripts/generate_benchmark_plots.py \
    --predictions /kaggle/working/test_predictions.json \
    --robustness /kaggle/working/robustness_results.json \
    --loto /kaggle/working/loto_results.json \
    --output_dir /kaggle/working/figures

!ls -lh /kaggle/working/*.pth /kaggle/working/*.json /kaggle/working/figures/*.png

In [ ]:
# Export trained backbone to ONNX
!python scripts/export_onnx.py \
    --weights /kaggle/working/dual_stream_calibrated.pth \
    --output /kaggle/working/models/dual_stream.onnx \
    --img_size 256

# Benchmark Inference Latency & FPS
!python scripts/benchmark_latency.py \
    --weights /kaggle/working/dual_stream_calibrated.pth \
    --batch_size 32 \
    --device cuda

# Render 4-Panel Interpretability Diagnostics
!python scripts/visualize_attention_maps.py \
    --checkpoint /kaggle/working/dual_stream_calibrated.pth \
    --output_dir /kaggle/working/figures/attention_maps \
    --n_samples 6